# From Curves to Circles: Understanding the Discrete Fourier Transform


Before tackling 2D drawings, let's understand the core idea in **1 dimension**: given a curve $f(x)$ sampled at discrete points, the **Discrete Fourier Transform (DFT)** decomposes it into a sum of sinusoids — each with its own **frequency**, **amplitude**, and **phase**.

Then the **Inverse DFT** reconstructs the original curve by adding those sinusoids back together.

---

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 12

## 2. The Big Idea

Any periodic signal sampled at $N$ points can be written as a sum of $N$ sinusoids:

$$f[n] = \sum_{k=0}^{N-1} A_k \cos\!\left(\frac{2\pi k n}{N} + \phi_k\right)$$

The DFT finds the amplitude $A_k$ and phase $\phi_k$ for each frequency $k$. This is **the Inverse DFT.** 

Let's see this in action with a simple example.

## DO ALL CODE LABELED "TODO":

## 3. Example 1 — A Simple Sum of Two Sine Waves

We'll create a signal that is the sum of two sine waves, then use DFT to recover them.

In [ ]:
N = 128  # number of sample points
n = np.arange(N) # 0,1,2...,127

# TODO: Our signal: two sine waves added together
# frequency 3 with amplitude 2, frequency 7 with amplitude 1 from the worksheet
# fill in the function below in signal
signal = 

plt.figure(figsize=(12, 3))
# TODO: plot the graph using plt.plot(x,y)




plt.scatter(n, signal, s=8, color='blue', zorder=5)
plt.title('Original Signal: sum of two sine waves (freq=3, freq=7)')
plt.xlabel('Sample index n')
plt.ylabel('f[n]')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Computing the DFT — By Hand (then with NumPy)

The DFT formula for frequency bin $k$ is:

$$X[k] = \sum_{n=0}^{N-1} f[n] \cdot e^{-i \, 2\pi k n / N}$$

This gives a **complex number** $X[k]$ for each frequency $k$. From it we extract:
- **Amplitude**: $A_k = \frac{|X[k]|}{N}$ (how strong this frequency is)
- **Phase**: $\phi_k = \text{atan2}(\text{Im}(X[k]),\, \text{Re}(X[k]))$ (where in its cycle it starts)

Let's implement this from scratch first, then verify with `np.fft.fft`.

In [ ]:
def my_dft(x):
    """Compute DFT from scratch — O(N^2), same logic as the p5.js version."""
    N = len(x)
    X = []
    for k in range(N):
        total = 0 + 0j
        for n in range(N):
            phi = 2 * np.pi * k * n / N
            total += x[n] * (np.cos(phi) - 1j * np.sin(phi))
        # Store amplitude and phase (dividing by N to normalize)
        re = total.real / N
        im = total.imag / N
        amp = np.sqrt(re**2 + im**2)
        phase = np.arctan2(im, re)
        X.append({'freq': k, 'amp': amp, 'phase': phase, 're': re, 'im': im})
    return X

# TODO: Compute DFT of our signal by calling my_dft
fourier =

# Show the frequency spectrum (amplitude vs frequency)
freqs = [f['freq'] for f in fourier]
amps = [f['amp'] for f in fourier]

plt.figure(figsize=(12, 3))
plt.stem(freqs[:N//2], amps[:N//2], linefmt='r-', markerfmt='ro', basefmt='gray')
plt.title('Frequency Spectrum (DFT amplitudes)')
plt.xlabel('Frequency k')
plt.ylabel('Amplitude')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Frequencies with significant amplitude:")
for f in fourier[:N//2]:
    if f['amp'] > 0.01:
        print(f"  freq={f['freq']}, amp={f['amp']:.4f}, phase={np.degrees(f['phase']):.1f}°")

The DFT correctly identifies two spikes at **frequency 3** (amplitude ≈ 2) and **frequency 7** (amplitude ≈ 1) — exactly what we put in!

## 5. Reconstruction — The Inverse DFT as a Sum of Sinusoids

Now the key insight: we can **reconstruct** the original signal by summing up cosines at each frequency:

$$f[n] = \sum_{k=0}^{N-1} A_k \cos\!\left(\frac{2\pi k n}{N} + \phi_k\right)$$

This is what the epicycles in the p5.js sketch are doing — each spinning circle corresponds to one term in this sum!

In [ ]:
def reconstruct(fourier, N, num_terms=None):
    """Reconstruct signal using the first `num_terms` frequencies (sorted by amplitude)."""
    if num_terms is None:
        num_terms = len(fourier)
    
    # Sort by amplitude (largest first) — just like the p5.js sketch does
    sorted_f = sorted(fourier, key=lambda f: f['amp'], reverse=True)
    
    result = np.zeros(N)
    for i in range(num_terms):
        f = sorted_f[i]
        for n in range(N):
            result[n] += f['amp'] * np.cos(2 * np.pi * f['freq'] * n / N + f['phase'])
    return result

# Full reconstruction (all terms)
reconstructed = reconstruct(fourier, N)

plt.figure(figsize=(12, 3))
plt.plot(n, signal, 'b-', linewidth=2, alpha=0.5, label='Original')
plt.plot(n, reconstructed, 'r--', linewidth=1.5, label='Reconstructed (all terms)')
plt.title('Original vs Reconstructed Signal')
plt.xlabel('Sample index n')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Max reconstruction error: {np.max(np.abs(signal - reconstructed)):.2e}")

Perfect reconstruction! The error is essentially zero (just floating-point noise).

## 6. Partial Reconstruction — Adding One Frequency at a Time

This is where it gets interesting. What if we only use the **top few frequencies** (by amplitude)? We get an **approximation** that gets better as we add more terms.

This is exactly what you see in the epicycle animation — each new circle refines the drawing.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

# TODO: Try different values for term_counts. If term_count = 1, then only one term is used to approximate.
# the more terms, the better the approximation.
# Fill in the list separated by commas of different term_count values
term_counts = [] 

for ax, num_terms in zip(axes.flat, term_counts):
    approx = reconstruct(fourier, N, num_terms=num_terms)
    ax.plot(n, signal, 'b-', alpha=0.3, linewidth=1, label='Original')
    ax.plot(n, approx, 'r-', linewidth=1.5, label=f'{num_terms} term{"s" if num_terms > 1 else ""}')
    ax.set_title(f'{num_terms} term{"s" if num_terms > 1 else ""}', fontweight='bold')
    ax.set_ylim(signal.min() - 0.5, signal.max() + 0.5)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

fig.suptitle('Approximation Quality vs Number of Frequency Terms', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Notice that with just **2 terms** we already have a nearly perfect approximation — because our signal was made of only 2 sine waves! The remaining terms have near-zero amplitude.

---

## 7. Example 2 — A More Complex Curve(Just run the code below)

Let's try something harder: a **square-ish wave** that isn't a simple sum of one or two sines. Now we'll really see how adding more frequency components gradually refines the approximation.

In [ ]:
N = 200
n = np.arange(N)
t = n / N  # normalized to [0, 1)

# A function with sharp features — needs many frequencies to represent
signal2 = np.sign(np.sin(2 * np.pi * 3 * t))  # square wave with 3 cycles

fourier2 = my_dft(signal2)

# Spectrum
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 3.5))

ax1.plot(n, signal2, 'b-', linewidth=1.5)
ax1.set_title('Square Wave (3 cycles)')
ax1.set_xlabel('Sample index')
ax1.grid(True, alpha=0.3)

ax2.stem([f['freq'] for f in fourier2[:N//2]],
         [f['amp'] for f in fourier2[:N//2]],
         linefmt='r-', markerfmt='ro', basefmt='gray')
ax2.set_title('Frequency Spectrum')
ax2.set_xlabel('Frequency k')
ax2.set_ylabel('Amplitude')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
term_counts = [1, 2, 3, 5, 10, 20, 50, N]

for ax, num_terms in zip(axes.flat, term_counts):
    approx = reconstruct(fourier2, N, num_terms=num_terms)
    ax.plot(n, signal2, 'b-', alpha=0.3, linewidth=1)
    ax.plot(n, approx, 'r-', linewidth=1.5)
    label = 'All' if num_terms == N else str(num_terms)
    ax.set_title(f'{label} terms', fontweight='bold')
    ax.set_ylim(-1.5, 1.5)
    ax.grid(True, alpha=0.3)

fig.suptitle('Square Wave: More Terms → Better Approximation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

The square wave needs **many** frequencies because of its sharp corners. Notice the characteristic **overshoot** near the jumps — this is called the [Gibbs phenomenon](https://en.wikipedia.org/wiki/Gibbs_phenomenon).



In [ ]:
N = 200
n = np.arange(N)
t = n / N

# A bumpy, asymmetric curve
signal3 = (  np.sin(2*np.pi*t)
           + 0.5 * np.sin(2*np.pi*3*t + 0.7)
           + 0.3 * np.cos(2*np.pi*7*t - 1.2)
           + 0.8 * np.sin(2*np.pi*2*t + 2.0)
           + 0.15 * np.cos(2*np.pi*13*t) )

fourier3 = my_dft(signal3)

plt.figure(figsize=(12, 3))
plt.plot(n, signal3, 'b-', linewidth=1.5)
plt.title('Arbitrary Bumpy Curve')
plt.xlabel('Sample index')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
term_counts = [1, 2, 3, 4, 5, 8, 15, N]

for ax, num_terms in zip(axes.flat, term_counts):
    approx = reconstruct(fourier3, N, num_terms=num_terms)
    ax.plot(n, signal3, 'b-', alpha=0.3, linewidth=1)
    ax.plot(n, approx, 'r-', linewidth=1.5)
    label = 'All' if num_terms == N else str(num_terms)
    ax.set_title(f'{label} terms', fontweight='bold')
    ax.set_ylim(signal3.min()-0.3, signal3.max()+0.3)
    ax.grid(True, alpha=0.3)

fig.suptitle('Bumpy Curve: Progressive Approximation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 8. Mystery Signal Challenge

Your group has been assigned a **mystery signal** — 100 sampled values generated from the sum of **3 cosines**, each with a specific **amplitude**, **frequency**, and **phase**.

Your job:
1. **Load** your group's mystery signal from GitHub
2. **Run the DFT** on it to get the frequency spectrum
3. **Read the spectrum** to identify the 3 frequencies and their amplitudes
4. **Read the phases** at those frequencies
5. **Write your own formula** using those values and plot it to verify it matches

$$\text{mystery}[n] = A_1 \cos\!\left(\frac{2\pi f_1 n}{100} + \phi_1\right) + A_2 \cos\!\left(\frac{2\pi f_2 n}{100} + \phi_2\right) + A_3 \cos\!\left(\frac{2\pi f_3 n}{100} + \phi_3\right)$$

**Hint:** The amplitudes are whole numbers (1–5), frequencies are integers, and phases are nice values like $0$, $\frac{\pi}{6}$, $\frac{\pi}{4}$, $\frac{\pi}{3}$, $\frac{\pi}{2}$, or $\pi$.

### Step 1: Load your group's mystery signal

**TODO:** Change `GROUP_NUMBER` to your assigned group number (0–9).

In [ ]:
import json
import urllib.request

# TODO: Change this to your group number (0-9)
GROUP_NUMBER = 0

# Load mystery signals from GitHub
# NOTE: Update this URL once you upload mystery_signals.json to your GitHub repo
url = 'https://raw.githubusercontent.com/YOUR_USERNAME/YOUR_REPO/main/mystery_signals.json'
response = urllib.request.urlopen(url)
all_signals = json.loads(response.read())

# Get your group's signal
mystery = np.array(all_signals[str(GROUP_NUMBER)])
N = len(mystery)  # should be 100
n = np.arange(N)

print(f'Loaded Group {GROUP_NUMBER} mystery signal: {N} samples')

# Plot the mystery signal
plt.figure(figsize=(12, 3))
plt.plot(n, mystery, 'b-', linewidth=1.5)
plt.scatter(n, mystery, s=10, color='blue', zorder=5)
plt.title(f'Mystery Signal (Group {GROUP_NUMBER})')
plt.xlabel('Sample index n')
plt.ylabel('f[n]')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Step 2: Run the DFT and examine the spectrum

**TODO:** Use `my_dft` (defined earlier in this notebook) to compute the DFT of your mystery signal. Then look at the frequency spectrum — you should see exactly **3 spikes**. Those are your 3 frequencies.

In [ ]:
# TODO: Compute the DFT of the mystery signal
mystery_fourier = 

# Plot the frequency spectrum
freqs = [f['freq'] for f in mystery_fourier]
amps = [f['amp'] for f in mystery_fourier]

plt.figure(figsize=(12, 4))
plt.stem(freqs[:N//2], amps[:N//2], linefmt='r-', markerfmt='ro', basefmt='gray')
plt.title(f'Mystery Signal Frequency Spectrum (Group {GROUP_NUMBER})')
plt.xlabel('Frequency k')
plt.ylabel('Amplitude')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print frequencies with significant amplitude
print('Frequencies with significant amplitude:')
print(f'{"freq":>6} {"amp":>10} {"phase (rad)":>12} {"phase (deg)":>12}')
print('-' * 44)
for f in mystery_fourier[:N//2]:
    if f['amp'] > 0.01:
        print(f"{f['freq']:>6} {f['amp']:>10.4f} {f['phase']:>12.4f} {np.degrees(f['phase']):>11.1f}°")

### Step 3: Identify the parameters

From the spectrum above, fill in the table:

| Component | Frequency ($f$) | Amplitude ($A$) | Phase ($\phi$) in radians |
|-----------|----------------|-----------------|---------------------------|
| 1st cosine |  |  |  |
| 2nd cosine |  |  |  |
| 3rd cosine |  |  |  |

**Phase cheat sheet** (match your phase values to the nearest one):

| Degrees | Radians | Approximate decimal |
|---------|---------|--------------------|
| 0°      | $0$     | 0.0000 |
| 30°     | $\pi/6$ | 0.5236 |
| 45°     | $\pi/4$ | 0.7854 |
| 60°     | $\pi/3$ | 1.0472 |
| 90°     | $\pi/2$ | 1.5708 |
| 180°    | $\pi$   | 3.1416 |

**Note:** Negative phase values are the same as their positive counterpart with a negative sign. For example, $-1.5708 = -\pi/2$.

### Step 4: Reconstruct the signal from your identified parameters

**TODO:** Fill in the amplitude, frequency, and phase values you found from the DFT. Then run the cell to see if your reconstruction matches the mystery signal!

In [ ]:
N = 100
n = np.arange(N)

# TODO: Fill in the values you found from the DFT
# Component 1
A1 =     # amplitude (integer 1-5)
f1 =     # frequency (integer)
phi1 =   # phase in radians (e.g., 0, np.pi/6, np.pi/4, np.pi/3, np.pi/2, np.pi)

# Component 2
A2 =     # amplitude
f2 =     # frequency
phi2 =   # phase

# Component 3
A3 =     # amplitude
f3 =     # frequency
phi3 =   # phase

# Build the reconstructed signal
my_signal = (  A1 * np.cos(2 * np.pi * f1 * n / N + phi1)
             + A2 * np.cos(2 * np.pi * f2 * n / N + phi2)
             + A3 * np.cos(2 * np.pi * f3 * n / N + phi3) )

# Plot comparison
plt.figure(figsize=(12, 4))
plt.plot(n, mystery, 'b-', linewidth=2, alpha=0.5, label='Mystery signal')
plt.plot(n, my_signal, 'r--', linewidth=1.5, label='Your reconstruction')
plt.scatter(n, mystery, s=10, color='blue', zorder=5, alpha=0.3)
plt.title(f'Mystery Signal vs Your Reconstruction (Group {GROUP_NUMBER})')
plt.xlabel('Sample index n')
plt.ylabel('f[n]')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Check accuracy
max_error = np.max(np.abs(mystery - my_signal))
print(f'\nMax error: {max_error:.6f}')
if max_error < 0.01:
    print('PERFECT! Your parameters match the mystery signal!')
elif max_error < 0.5:
    print('Close! Check your phase values — they might be slightly off.')
else:
    print('Not quite — re-examine the spectrum and try again.')

### Step 5: Write your answer

**TODO:** Once your plot matches, write the formula for your mystery signal below:

$$f[n] = \rule{1cm}{0.15mm} \cdot \cos\!\left(\frac{2\pi \cdot \rule{0.5cm}{0.15mm} \cdot n}{100} + \rule{1cm}{0.15mm}\right) + \rule{1cm}{0.15mm} \cdot \cos\!\left(\frac{2\pi \cdot \rule{0.5cm}{0.15mm} \cdot n}{100} + \rule{1cm}{0.15mm}\right) + \rule{1cm}{0.15mm} \cdot \cos\!\left(\frac{2\pi \cdot \rule{0.5cm}{0.15mm} \cdot n}{100} + \rule{1cm}{0.15mm}\right)$$